# 03 — HTTP avec httpx

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- envoyer des requêtes HTTP (GET/POST/PUT/DELETE) avec `httpx`;
- gérer paramètres, en-têtes, JSON, formulaires et upload de fichiers;
- utiliser un `Client` avec pooling et session persistante;
- lire des réponses streaming et télécharger des gros fichiers;
- passer à l'**async** avec `httpx.AsyncClient`;
- implémenter **retries** et **timeouts** robustes;
- authentification : Basic, Bearer, API Key, OAuth2 (client credentials);


## Prérequis — ce que vous devez maîtriser

Ce notebook fait partie des **Side tracks**. Il est conçu pour être autonome, à condition d'avoir suivi le niveau **Intermédiaire** de la formation principale. Vous devez donc être à l'aise avec :

- les bases HTTP (méthodes, codes, headers, JSON);
- type hints et contexts managers;
- async/await (pour la section async);

Ce que ce notebook **ne couvrira pas** (ou seulement en mention) :

- WebSockets (voir `04_Websockets.ipynb`);
- scraping complet (voir BeautifulSoup / Scrapy);


> **Note.** `httpx` est le successeur de `requests` : même API mais avec support **sync + async**, HTTP/2, typing complet, et gestion moderne des timeouts.


## Plan

1. Installation et import
2. Première requête GET
3. Query parameters
4. Headers
5. Codes de statut et `raise_for_status`
6. JSON : envoi et réception
7. Formulaires et fichiers
8. Le `Client` : pool de connexions
9. Timeouts
10. Retries avec `tenacity`
11. Authentification
12. Streaming de la réponse
13. Upload de gros fichiers
14. HTTPX async
15. Cas pratique : scraping JSON paginé
16. Synthèse
17. Exercices


---


## 1. Installation et import

```bash
pip install httpx
pip install httpx[http2]  # optionnel pour HTTP/2
```


In [ ]:
import httpx
httpx.__version__

---


## 2. Première requête GET


In [ ]:
resp = httpx.get('https://httpbin.org/get')
resp.status_code

In [ ]:
resp.headers['content-type']

In [ ]:
resp.json()

In [ ]:
resp.text[:200]

---


## 3. Query parameters


In [ ]:
resp = httpx.get('https://httpbin.org/get', params={'q': 'python', 'page': 2})
resp.url

---


## 4. Headers


In [ ]:
resp = httpx.get('https://httpbin.org/get',
    headers={'User-Agent': 'MonApp/1.0', 'Accept': 'application/json'})
resp.json()['headers']

---


## 5. Codes de statut et `raise_for_status`


In [ ]:
resp = httpx.get('https://httpbin.org/status/404')
resp.status_code

In [ ]:
try:
    resp.raise_for_status()
except httpx.HTTPStatusError as e:
    print('Erreur HTTP :', e.response.status_code)

---


## 6. JSON : envoi et réception


In [ ]:
resp = httpx.post('https://httpbin.org/post', json={'nom': 'Alice', 'age': 30})
resp.json()['json']

---


## 7. Formulaires et fichiers


In [ ]:
# Form-urlencoded
resp = httpx.post('https://httpbin.org/post', data={'user': 'alice'})
resp.json()['form']

In [ ]:
# Upload
# files = {'fichier': open('image.png', 'rb')}
# httpx.post('https://httpbin.org/post', files=files)

---


## 8. Le `Client` : pool de connexions

Créer un `Client` (équivalent de `Session` en `requests`) permet de réutiliser les connexions TCP/TLS, partager les cookies et headers par défaut. **Indispensable** si vous faites plus de 3 requêtes à la même origine.


In [ ]:
with httpx.Client(base_url='https://httpbin.org',
                  headers={'User-Agent': 'MonApp/1.0'},
                  timeout=10.0) as client:
    r1 = client.get('/get')
    r2 = client.get('/headers')
    print(r1.status_code, r2.status_code)

---


## 9. Timeouts

`httpx` force un timeout par défaut (5 s connect, 5 s read, 5 s write, 5 s pool). C'est un gros progrès par rapport à `requests` qui n'en avait aucun par défaut.


In [ ]:
t = httpx.Timeout(connect=5.0, read=30.0, write=5.0, pool=5.0)
httpx.Client(timeout=t)

**Toujours** fixer un timeout en production. Sans timeout, un serveur lent peut bloquer votre process indéfiniment.


---


## 10. Retries avec `tenacity`

`httpx` ne fait pas de retries par défaut. On utilise `tenacity` pour ça.


In [ ]:
# pip install tenacity
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=0.5, min=1, max=10),
       retry=retry_if_exception_type((httpx.TimeoutException, httpx.ConnectError)))
def get_robuste(url: str) -> httpx.Response:
    resp = httpx.get(url, timeout=5.0)
    resp.raise_for_status()
    return resp

---


## 11. Authentification


### Basic Auth


In [ ]:
resp = httpx.get('https://httpbin.org/basic-auth/alice/secret',
    auth=('alice', 'secret'))
resp.json()

### Bearer token


In [ ]:
resp = httpx.get('https://httpbin.org/headers',
    headers={'Authorization': 'Bearer mon-token'})
resp.json()['headers']['Authorization']

### API Key dans un header custom


In [ ]:
client = httpx.Client(headers={'X-API-Key': 'abcd'})
# toutes les requêtes du client porteront l'en-tête

### OAuth2 client credentials (pattern)


In [ ]:
def get_token(client_id: str, client_secret: str, token_url: str) -> str:
    resp = httpx.post(token_url, data={
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
    })
    resp.raise_for_status()
    return resp.json()['access_token']

---


## 12. Streaming de la réponse

Pour un gros téléchargement, utilisez `client.stream(...)` et itérez.


In [ ]:
# with httpx.Client() as client:
#     with client.stream('GET', 'https://example.com/gros.bin') as r:
#         with open('gros.bin', 'wb') as f:
#             for chunk in r.iter_bytes(chunk_size=65536):
#                 f.write(chunk)

---


## 13. Upload de gros fichiers


In [ ]:
# files = {'fichier': ('nom.bin', open('gros.bin', 'rb'), 'application/octet-stream')}
# httpx.post('https://httpbin.org/post', files=files)

---


## 14. HTTPX async

Même API, mais avec `AsyncClient` et `await`. Idéal pour paralléliser des dizaines ou centaines de requêtes.


In [ ]:
import asyncio
import httpx

async def fetch(url: str) -> int:
    async with httpx.AsyncClient() as client:
        r = await client.get(url)
        return r.status_code

# asyncio.run(fetch('https://httpbin.org/get'))

In [ ]:
async def fetch_all(urls: list[str]) -> list[int]:
    async with httpx.AsyncClient() as client:
        tasks = [client.get(u) for u in urls]
        resps = await asyncio.gather(*tasks)
        return [r.status_code for r in resps]

Avec `asyncio.gather`, N requêtes parallèles se font en **temps d'une seule** (à la limite de votre réseau).


---


## 15. Cas pratique : scraping d'une API paginée


In [ ]:
def tout_recuperer(base: str) -> list[dict]:
    resultats: list[dict] = []
    with httpx.Client(base_url=base, timeout=10.0) as client:
        url = '/items?page=1'
        while url:
            r = client.get(url)
            r.raise_for_status()
            data = r.json()
            resultats.extend(data['items'])
            url = data.get('next')
    return resultats

---


## Synthèse

| Notion | À retenir |
|---|---|
| `httpx.get/post/...` | fonctions simples pour requêtes one-shot |
| `httpx.Client` | pool de connexions, session, à utiliser dans 99 % des cas |
| `timeout` | par défaut 5s partout — **ne jamais le désactiver** |
| `raise_for_status` | lève si `>= 400` |
| `json=...` | envoi JSON (encode + Content-Type automatique) |
| `params=...` | query string |
| `files=...` | upload multipart |
| `AsyncClient` | miroir async pour `asyncio.gather` |
| `stream` | pour gros téléchargements |


### Règles à retenir

- **Toujours un `Client` en context manager** pour plusieurs requêtes.
- **Toujours un timeout** (httpx en met par défaut, ne pas le retirer).
- **`raise_for_status`** juste après la requête pour détecter tôt.
- **`tenacity`** pour retries, pas de boucle `while` manuelle.
- **`AsyncClient`** pour paralléliser, pas de thread pool.


---


## Exercices

Pour chaque exercice : écrivez une **fonction typée**, testez-la, puis comparez avec la correction. N'ouvrez pas la correction avant d'avoir vraiment cherché.


### Exercice 1 — Récupérer un JSON

Écrivez `get_json(url)` qui renvoie le JSON d'une URL, ou lève si le statut n'est pas 2xx.


In [ ]:
import httpx

def get_json(url: str) -> dict:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import httpx

def get_json(url: str) -> dict:
    resp = httpx.get(url, timeout=10.0)
    resp.raise_for_status()
    return resp.json()
```

</details>


### Exercice 2 — POST JSON

Écrivez `poster(url, payload)` qui envoie `payload` en JSON et renvoie la réponse décodée.


In [ ]:
def poster(url: str, payload: dict) -> dict:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def poster(url: str, payload: dict) -> dict:
    resp = httpx.post(url, json=payload, timeout=10.0)
    resp.raise_for_status()
    return resp.json()
```

</details>


### Exercice 3 — Client réutilisable

Écrivez une fonction `ping_plusieurs(urls)` qui utilise un **unique** `Client` et renvoie un dict `{url: status_code}`.


In [ ]:
def ping_plusieurs(urls: list[str]) -> dict[str, int]:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def ping_plusieurs(urls: list[str]) -> dict[str, int]:
    with httpx.Client(timeout=5.0) as client:
        return {u: client.get(u).status_code for u in urls}
```

</details>


### Exercice 4 — Version async

Écrivez `async_ping_plusieurs(urls)` version asynchrone avec `AsyncClient` et `asyncio.gather`.


In [ ]:
import asyncio

async def async_ping_plusieurs(urls: list[str]) -> dict[str, int]:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import httpx

async def async_ping_plusieurs(urls: list[str]) -> dict[str, int]:
    async with httpx.AsyncClient(timeout=5.0) as client:
        resps = await asyncio.gather(*(client.get(u) for u in urls))
        return {u: r.status_code for u, r in zip(urls, resps)}
```

</details>


---


## Ressources externes

- [Documentation httpx](https://www.python-httpx.org/)
- [httpx Advanced Usage](https://www.python-httpx.org/advanced/)
- [tenacity](https://tenacity.readthedocs.io/)
